# Last.fm Album Recommendations

Similar-album recommendations via Last.fm's `track.getSimilar` API.

Compare three ways of picking the seed track from the album:

- **(a) Top listener** — the track with the most Last.fm listeners on the album
- **(b) Random** — a random track from the album tracklist
- **(c) Top-N tracks** — use the `top_n` tracks (default 3) with the most listeners, aggregate recommendations by vote count; falls back to **(a)** if nothing is found

Set your API key in `bench/.env`:

```
LASTFM_API_KEY=your_key_here
```

In [1]:
import importlib

import pandas as pd

import lastfm_albums
importlib.reload(lastfm_albums)
from lastfm_albums import get_album_info, recommend_album

In [3]:
N_RECS = 5
FETCH_FLOOR = 10
TOP_N = 3
RANDOM_SEED = 42

## Seed album

In [4]:
SEED_ALBUM = "Take Care"
SEED_ARTIST = "Drake"

In [6]:
seed = get_album_info(SEED_ARTIST, SEED_ALBUM)
pd.Series({k: seed[k] for k in seed if k != "tracks"})

key                                   drake::take care
artist                                           Drake
album                                        Take Care
mbid              02f7e348-b1ae-4c4e-a057-b97f1f56e53c
playcount                                     16203374
listeners                                       567881
url          https://www.last.fm/music/Drake/Take+Care
tags                  [rap, hip-hop, rnb, 2011, drake]
dtype: object

## Seed-track strategies

Run one cell at a time. Each calls `recommend_album()` directly.

In [7]:
# (a) Top-listener track
_, seed_track_a, recs_a, _ = recommend_album(
    SEED_ALBUM,
    artist=SEED_ARTIST,
    n_recs=N_RECS,
    fetch_floor=FETCH_FLOOR,
    track_selection="top_listener",
)
recs_a

,key,artist,album,url,similarity_score,seed_track,matched_via
0,j. cole::cole world: the sideline story,J. Cole,Cole World: The Sideline Story,https://www.last.fm/music/J.+Cole/Cole+World:+...,0.377067,Headlines,track 'Work Out' similar to 'Headlines'
1,drake & 21 savage::her loss,Drake & 21 Savage,Her Loss,https://www.last.fm/music/Drake+&+21+Savage/He...,0.356263,Headlines,track 'Hours in Silence' similar to 'Headlines'
2,kanye west::yeezus,Kanye West,Yeezus,https://www.last.fm/music/Kanye+West/Yeezus,0.318118,Headlines,track 'Bound 2' similar to 'Headlines'
3,kanye west::my beautiful dark twisted fantasy ...,Kanye West,My Beautiful Dark Twisted Fantasy (Deluxe Edit...,https://www.last.fm/music/Kanye+West/My+Beauti...,0.289621,Headlines,track 'Devil in a New Dress (feat. Rick Ross)'...


In [ ]:
# (b) Random track
_, seed_track_b, recs_b, _ = recommend_album(
    SEED_ALBUM,
    artist=SEED_ARTIST,
    n_recs=N_RECS,
    fetch_floor=FETCH_FLOOR,
    track_selection="random",
    random_seed=RANDOM_SEED,
    clear_cache=False,
)
recs_b[["artist", "album", "matched_via"]]

,key,artist,album,url,similarity_score,seed_track,matched_via
0,metro boomin::heroes & villains,Metro Boomin,HEROES & VILLAINS,https://www.last.fm/music/Metro+Boomin/HEROES+...,0.782101,Crew Love,track 'Creepin' (with The Weeknd & 21 Savage)'...
1,nav::nav,NAV,NAV,https://www.last.fm/music/NAV/NAV,0.749463,Crew Love,track 'Some Way' similar to 'Crew Love'
2,the weeknd::after hours,The Weeknd,After Hours,https://www.last.fm/music/The+Weeknd/After+Hours,0.681483,Crew Love,track 'Snowchild' similar to 'Crew Love'
3,kanye west::the life of paul,Kanye West,The Life Of Paul,https://www.last.fm/music/Kanye+West/The+Life+...,0.405488,Crew Love,track 'FML' similar to 'Crew Love'
4,future::we still don't trust you,Future,WE STILL DON'T TRUST YOU,https://www.last.fm/music/Future/WE+STILL+DON%...,0.399409,Crew Love,track 'All To Myself' similar to 'Crew Love'


In [19]:
# (c) Top-N tracks by listeners
_, seed_track_c, recs_c, used_fallback_c = recommend_album(
    SEED_ALBUM,
    artist=SEED_ARTIST,
    n_recs=10,
    fetch_floor=FETCH_FLOOR,
    track_selection="top_n_tracks",
    top_n=TOP_N,
    clear_cache=False,
)
print(f"Needed to user fallback? {used_fallback_c}")
recs_c[["artist", "album", "seed_track", "matched_via", "similarity_score"]]

Needed to user fallback? False


,artist,album,seed_track,matched_via,similarity_score
0,Metro Boomin,HEROES & VILLAINS,"Headlines, Take Care, Crew Love",from 1 of 3 top tracks (Crew Love),0.782101
1,NAV,NAV,"Headlines, Take Care, Crew Love",from 1 of 3 top tracks (Crew Love),0.749463
2,The Weeknd,After Hours,"Headlines, Take Care, Crew Love",from 1 of 3 top tracks (Crew Love),0.681483
3,Kanye West,The Life Of Paul,"Headlines, Take Care, Crew Love",from 1 of 3 top tracks (Crew Love),0.405488
4,Future,WE STILL DON'T TRUST YOU,"Headlines, Take Care, Crew Love",from 1 of 3 top tracks (Crew Love),0.399409
5,Future,WE DON'T TRUST YOU,"Headlines, Take Care, Crew Love",from 1 of 3 top tracks (Crew Love),0.391861
6,J. Cole,Cole World: The Sideline Story,"Headlines, Take Care, Crew Love",from 1 of 3 top tracks (Headlines),0.377067
7,Drake & 21 Savage,Her Loss,"Headlines, Take Care, Crew Love",from 1 of 3 top tracks (Headlines),0.356263
8,Kanye West,Yeezus,"Headlines, Take Care, Crew Love",from 1 of 3 top tracks (Headlines),0.318118
9,Lil Uzi Vert,Luv Is Rage 2 (Deluxe),"Headlines, Take Care, Crew Love",from 1 of 3 top tracks (Crew Love),0.306776


In [9]:
# Overlap between (a) and (b)
recs_a.merge(recs_b, on="key", suffixes=("_top_listener", "_random"))

,key,artist_top_listener,album_top_listener,url_top_listener,similarity_score_top_listener,seed_track_top_listener,matched_via_top_listener,artist_random,album_random,url_random,similarity_score_random,seed_track_random,matched_via_random
